# 🏦 Deal Review Pipeline
### LangGraph + LangChain + Ollama

Three agents in sequence: **Extract → Compliance → Risk**

**Before running**, open a terminal:
```bash
ollama serve
ollama pull gemma4:e4b 
```

# 🏗️ Deal Review Pipeline — Architecture

A **sequential multi-agent pipeline** built with **LangGraph** and **Ollama**. Three specialized ReAct agents run in a fixed order, each passing structured data to the next via a shared typed `State`. No vector database — the PDF text flows directly through the pipeline.

---

```
                                  DEAL REVIEW PIPELINE
 ─────────────────────────────────────────────────────────────────────────────────────

  📄 PDF
     │
     │  deal_text
     ▼
  ┌─────────────────────┐       ┌─────────────────────┐       ┌─────────────────────┐
  │   🟢  AGENT 1        │       │   🔵  AGENT 2        │       │   🔴  AGENT 3        │
  │      Extractor      │       │     Compliance      │       │    Risk Scorer      │
  │                     │       │                     │       │                     │
  │  tools              │       │  tools              │       │  tools              │
  │  · (none)           │       │  · get_rules()      │       │  · credit_risk()    │
  │  · text in prompt   │       │  · check_jurisd()   │       │  · concentr_risk()  │
  │                     │       │                     │       │                     │
  │  out: State.terms   │──────▶│  out: State.verdict │──────▶│  out: State.report  │
  └─────────────────────┘       └─────────────────────┘       └──────────┬──────────┘
                                                                          │
                                                                          ▼
                                                               ┌─────────────────────┐
                                                               │   RECOMMENDATION    │
                                                               │                     │
                                                               │  ✅  APPROVE         │
                                                               │  ⚠️   CONDITIONAL    │
                                                               │  ❌  REJECT          │
                                                               └─────────────────────┘
 ─────────────────────────────────────────────────────────────────────────────────────
```

| | Agent 1 — Extractor | Agent 2 — Compliance | Agent 3 — Risk Scorer |
|---|---|---|---|
| **Input** | Raw PDF text | `State.terms` | `State.terms` + `State.verdict` |
| **Tools** | None — text is in the prompt | `get_rules`, `check_jurisdiction` | `credit_risk`, `concentration_risk` |
| **Output** | Structured JSON (amount, LTV, DTI, rate, credit score…) | Pass / Fail / Conditional + failed rules list | Risk score 1–100 + executive memo |
| **Role** | Parse the document | Enforce policy thresholds | Assess financial exposure |

## Step 1 — Install

In [ ]:
%pip install -q langchain langchain-ollama langgraph pypdf pydantic fpdf2


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2 — Imports

In [ ]:
import json, re, uuid
from typing import TypedDict, Optional
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from pypdf import PdfReader
print('OK')

## Step 3 — Setup
One LLM + one rule-set, shared by all agents.

In [ ]:
llm = ChatOllama(model='gemma4:e4b', temperature=0, num_ctx=4096)

RULES = {
    'max_loan_usd':          50_000_000,
    'max_ltv':                0.80,
    'min_credit_score':       620,
    'max_dti':                0.43,
    'max_rate_pct':           18.0,
    'escalation_usd':         10_000_000,
    'blocked_jurisdictions': ['OFAC', 'FATF-blacklist'],
}

print(llm.invoke('say OK').content)

## Step 4 — Tools
Functions the LLM can call. The docstring tells it when to use each one.

In [ ]:
# Agent 2 tools
@tool
def get_rules() -> str:
    'Return all regulatory thresholds.'
    return str(RULES)

@tool
def check_jurisdiction(jurisdiction: str) -> str:
    'Check if a jurisdiction is blocked or clear.'
    blocked = RULES['blocked_jurisdictions']
    return 'RESTRICTED' if any(b.upper() in jurisdiction.upper() for b in blocked) else 'CLEAR'

# Agent 3 tools
@tool
def credit_risk(credit_score: int, ltv: float, dti: float, rate: float) -> str:
    'Calculate credit risk score 1-100 (higher = riskier).'
    s = int((850-credit_score)/850*40 + ltv*30 + min(dti/0.43,1)*20 + min(rate/18,1)*10)
    return json.dumps({'score': s, 'level': 'LOW' if s<25 else 'MEDIUM' if s<50 else 'HIGH' if s<75 else 'CRITICAL'})

@tool
def concentration_risk(amount_usd: float) -> str:
    'Calculate what % of the $500M portfolio this deal represents.'
    pct = amount_usd / 500_000_000
    return json.dumps({'pct': round(pct*100,1), 'breached': pct > 0.10})

print('tools OK')

## Step 5 — Three Agents

In [ ]:
AGENT1 = create_react_agent(
    llm, tools=[],
    prompt=(
        'Extract as JSON: deal_id, deal_type, amount_usd, borrower, lender,'
        ' jurisdiction, interest_rate_pct, maturity_months, ltv_ratio,'
        ' credit_score, debt_to_income, covenants (list), collateral.'
        ' No markdown. Null for missing fields.'
    ))

AGENT2 = create_react_agent(
    llm, tools=[get_rules, check_jurisdiction],
    prompt=(
        'Check deal terms against RULES. Return JSON: deal_id,'
        ' overall_status (PASS/FAIL/CONDITIONAL), kyc_cleared, aml_cleared,'
        ' requires_escalation, failed_rules (list), notes. No markdown.'
    ))

AGENT3 = create_react_agent(
    llm, tools=[credit_risk, concentration_risk],
    prompt=(
        'Return JSON: deal_id, final_recommendation (APPROVE/REJECT/CONDITIONAL APPROVE),'
        ' credit_risk_score (1-100), overall_risk_level (LOW/MEDIUM/HIGH/CRITICAL),'
        ' escalation_required, key_risks (list), executive_memo (2-3 sentences).'
        ' No markdown.'
    ))

print('agents OK')

## Step 6 — Pipeline
Simple fixed sequence — no orchestrator, no MemorySaver needed.

In [9]:
# Minimal state — just the data passing between agents
class State(TypedDict):
    deal_id:  str
    deal_text: str
    terms:    Optional[dict]
    verdict:  Optional[dict]
    report:   Optional[dict]

# Parse JSON from LLM output (handles markdown fences too)
def parse(text):
    for fn in [
        lambda t: json.loads(t),
        lambda t: json.loads(re.search(r'```(?:json)?\s*([\s\S]+?)```', t).group(1)),
        lambda t: json.loads(re.search(r'\{[\s\S]+\}', t).group(0)),
    ]:
        try: return fn(text)
        except: pass
    raise ValueError(text[:200])

# Run an agent, return its response text
def ask(agent, msg):
    return agent.invoke({'messages': [{'role': 'user', 'content': msg}]})['messages'][-1].content

# The three nodes
def node_extract(s: State):
    print('[1/3] Extracting terms...')
    d = parse(ask(AGENT1, f"Deal ID: {s['deal_id']}\n\n{s['deal_text']}\n\nExtract as JSON."))
    d['deal_id'] = s['deal_id']
    return {'terms': d}

def node_compliance(s: State):
    print('[2/3] Compliance check...')
    d = parse(ask(AGENT2, f"Terms: {json.dumps(s['terms'])}\n\nCheck against RULES. Return JSON."))
    d['deal_id'] = s['deal_id']
    return {'verdict': d}

def node_risk(s: State):
    print('[3/3] Risk scoring...')
    d = parse(ask(AGENT3, f"Terms: {json.dumps(s['terms'])}\nCompliance: {json.dumps(s['verdict'])}\n\nReturn JSON."))
    d['deal_id'] = s['deal_id']
    return {'report': d}

# Wire graph — fixed sequence, direct edges, no routing logic
g = StateGraph(State)
g.add_node('extract',    node_extract)
g.add_node('compliance', node_compliance)
g.add_node('risk',       node_risk)

g.add_edge(START,        'extract')
g.add_edge('extract',    'compliance')
g.add_edge('compliance', 'risk')
g.add_edge('risk',        END)

pipeline = g.compile()
print('pipeline ready')


pipeline ready


## Step 7 — Run
Using the generated `test_deal.pdf`.

In [ ]:
reader    = PdfReader('test_deal.pdf')
DEAL_TEXT = chr(10).join(p.extract_text() or '' for p in reader.pages)

DEAL_ID = f'DEAL-{uuid.uuid4().hex[:6].upper()}'

result = pipeline.invoke(
    {'deal_id': DEAL_ID, 'deal_text': DEAL_TEXT,
     'terms': None, 'verdict': None, 'report': None}
)
print('Done!')

## Step 8 — Results

In [11]:
t = result.get('terms',   {})
v = result.get('verdict', {})
r = result.get('report',  {})

print(f"DEAL       : {result['deal_id']}")
print(f"AMOUNT     : ${t.get('amount_usd', 0):,.0f}")
print(f"COMPLIANCE : {v.get('overall_status')}")
print(f"RISK LEVEL : {r.get('overall_risk_level')}")
print(f"RISK SCORE : {r.get('credit_risk_score')}/100")
print(f"ESCALATION : {r.get('escalation_required')}")
print(f"VERDICT    : {r.get('final_recommendation')}")
print(f"\nMEMO: {r.get('executive_memo')}")


DEAL       : DEAL-441E02
AMOUNT     : $15,000,000
COMPLIANCE : CONDITIONAL
RISK LEVEL : MEDIUM
RISK SCORE : 44/100
ESCALATION : True
VERDICT    : CONDITIONAL APPROVE

MEMO: The deal presents solid financials with a favorable LTV (0.65) and low concentration risk (3.0%). However, the loan amount exceeds the internal escalation limit, necessitating manual review and executive sign-off before final approval.
